# Agent 2 — Notebook 03 FINAL

## Parsing Quality Evaluation and Human-in-the-Loop Review

This notebook evaluates the PostgreSQL records committed by Notebook 02 and adds a controlled Human-in-the-Loop workflow.

It includes:

- automatic checks for question numbers, marks, text completeness, header/footer noise, duplicates, code/visual context, QP/MS links, mark consistency, legacy status, and MS guidance;
- a persistent review queue;
- approve, correct, reject, legacy-disable, and manual MS-link actions;
- reviewer audit history;
- correction memory for future parser improvement;
- safe promotion to retrieval only after approval.

### Status flow

```text
auto_approved → passed automatic checks, still not published
needs_review  → human decision required
approved      → retrieval/indexing allowed
corrected     → retrieval/indexing allowed
rejected      → excluded
legacy_disabled → excluded
```


## 1. Install dependencies


In [1]:
%pip install -q pandas python-dotenv ipywidgets "sqlalchemy>=2.0" "psycopg[binary]>=3.1"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and configuration


In [2]:
from __future__ import annotations

import hashlib
import json
import os
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, clear_output, display
from dotenv import load_dotenv
from sqlalchemy import (
    Boolean, CheckConstraint, Column, DateTime, ForeignKey, Index,
    Integer, MetaData, String, Table, Text, UniqueConstraint,
    create_engine, delete, func, insert, select, update,
)
from sqlalchemy.dialects.postgresql import JSONB, UUID, insert as pg_insert
from sqlalchemy.orm import Session

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebooks", "notebook"} else cwd
OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")
DATABASE_URL = os.getenv("AGENT2_DATABASE_URL", "").strip()
if not DATABASE_URL:
    raise RuntimeError("AGENT2_DATABASE_URL is missing from Agent2/.env")

EVALUATOR_VERSION = "agent2-quality-hitl-v1.0.0"
SAVE_EVALUATION_TO_DB = True
RESET_EXISTING_DECISIONS = False
DEFAULT_REVIEWER = os.getenv("AGENT2_REVIEWER", "human_reviewer")

MIN_QUESTION_TEXT_LENGTH = 15
SUSPICIOUS_SHORT_LENGTH = 25

HEADER_FOOTER_PATTERNS = [
    r"physicsandmathstutor\.com",
    r"do not write outside",
    r"turn over",
    r"copyright information",
    r"answer all questions in the spaces provided",
    r"there are no questions printed on this page",
]

def utc_now() -> datetime:
    return datetime.now(timezone.utc)

def make_json_safe(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [make_json_safe(v) for v in value]
    if isinstance(value, (uuid.UUID, Path)):
        return str(value)
    if isinstance(value, datetime):
        return value.isoformat()
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if hasattr(value, "item"):
        try:
            return make_json_safe(value.item())
        except Exception:
            pass
    return str(value)

print(f"Project root:      {PROJECT_ROOT}")
print(f"Evaluator version: {EVALUATOR_VERSION}")
print(f"Reviewer:          {DEFAULT_REVIEWER}")


Project root:      C:\Users\hp\EDTECH\Agent2
Evaluator version: agent2-quality-hitl-v1.0.0
Reviewer:          human_reviewer


## 3. Connect and reflect Notebook 02 tables


In [3]:
engine = create_engine(DATABASE_URL, pool_pre_ping=True, future=True)
source_metadata = MetaData()

topics = Table("assessment_topical_topics", source_metadata, autoload_with=engine)
documents = Table("assessment_topical_documents", source_metadata, autoload_with=engine)
questions = Table("assessment_topical_questions", source_metadata, autoload_with=engine)
mark_scheme_entries = Table("assessment_topical_mark_scheme_entries", source_metadata, autoload_with=engine)
question_ms_links = Table("assessment_topical_question_mark_scheme_links", source_metadata, autoload_with=engine)
parsing_issues = Table("assessment_topical_parsing_issues", source_metadata, autoload_with=engine)

with engine.connect() as connection:
    connection.exec_driver_sql("SELECT 1")

print("PostgreSQL connection and table reflection successful.")


PostgreSQL connection and table reflection successful.


## 4. Create Human-in-the-Loop tables


In [4]:
review_metadata = MetaData()

quality_runs = Table(
    "assessment_topical_quality_runs",
    review_metadata,
    Column("id", UUID(as_uuid=True), primary_key=True),
    Column("evaluator_version", String(100), nullable=False),
    Column("status", String(30), nullable=False),
    Column("started_at", DateTime(timezone=True), nullable=False),
    Column("completed_at", DateTime(timezone=True)),
    Column("counts", JSONB, nullable=False, default=dict),
    Column("error_message", Text),
)

quality_checks = Table(
    "assessment_topical_quality_checks",
    review_metadata,
    Column("id", UUID(as_uuid=True), primary_key=True),
    Column("run_id", UUID(as_uuid=True), ForeignKey("assessment_topical_quality_runs.id", ondelete="CASCADE"), nullable=False),
    Column("entity_type", String(30), nullable=False),
    Column("entity_id", UUID(as_uuid=True), nullable=False),
    Column("pair_key", String(180), nullable=False),
    Column("check_code", String(100), nullable=False),
    Column("passed", Boolean, nullable=False),
    Column("severity", String(20), nullable=False),
    Column("message", Text, nullable=False),
    Column("details", JSONB, nullable=False, default=dict),
    Column("created_at", DateTime(timezone=True), nullable=False),
    UniqueConstraint("run_id", "entity_type", "entity_id", "check_code", name="uq_topical_quality_check_run_entity_code"),
    CheckConstraint("severity IN ('info','warning','critical')", name="ck_topical_quality_check_severity"),
    Index("ix_topical_quality_checks_entity", "entity_type", "entity_id"),
)

review_queue = Table(
    "assessment_topical_review_queue",
    review_metadata,
    Column("id", UUID(as_uuid=True), primary_key=True),
    Column("queue_uid", String(220), nullable=False, unique=True),
    Column("quality_run_id", UUID(as_uuid=True), ForeignKey("assessment_topical_quality_runs.id", ondelete="SET NULL")),
    Column("entity_type", String(30), nullable=False),
    Column("entity_id", UUID(as_uuid=True), nullable=False),
    Column("pair_key", String(180), nullable=False),
    Column("priority", Integer, nullable=False, default=0),
    Column("status", String(30), nullable=False),
    Column("reasons", JSONB, nullable=False, default=list),
    Column("assigned_to", String(120)),
    Column("reviewer_note", Text),
    Column("created_at", DateTime(timezone=True), nullable=False),
    Column("updated_at", DateTime(timezone=True), nullable=False),
    CheckConstraint(
        "status IN ('auto_approved','needs_review','approved','corrected','rejected','legacy_disabled')",
        name="ck_topical_review_queue_status",
    ),
    Index("ix_topical_review_queue_status_priority", "status", "priority"),
)

human_reviews = Table(
    "assessment_topical_human_reviews",
    review_metadata,
    Column("id", UUID(as_uuid=True), primary_key=True),
    Column("queue_id", UUID(as_uuid=True), ForeignKey("assessment_topical_review_queue.id", ondelete="SET NULL")),
    Column("entity_type", String(30), nullable=False),
    Column("entity_id", UUID(as_uuid=True), nullable=False),
    Column("action", String(40), nullable=False),
    Column("reviewer", String(120), nullable=False),
    Column("note", Text),
    Column("old_values", JSONB, nullable=False, default=dict),
    Column("new_values", JSONB, nullable=False, default=dict),
    Column("created_at", DateTime(timezone=True), nullable=False),
    CheckConstraint(
        "action IN ('approve','bulk_approve','correct','reject','legacy_disable','manual_link','resolve_issue')",
        name="ck_topical_human_review_action",
    ),
    Index("ix_topical_human_reviews_entity", "entity_type", "entity_id"),
)

correction_memory = Table(
    "assessment_topical_correction_memory",
    review_metadata,
    Column("id", UUID(as_uuid=True), primary_key=True),
    Column("memory_key", String(220), nullable=False, unique=True),
    Column("correction_type", String(60), nullable=False),
    Column("pair_key", String(180), nullable=False),
    Column("source_entity_id", UUID(as_uuid=True), nullable=False),
    Column("original_value", JSONB, nullable=False, default=dict),
    Column("corrected_value", JSONB, nullable=False, default=dict),
    Column("reviewer", String(120), nullable=False),
    Column("confirmations", Integer, nullable=False, default=1),
    Column("approved_for_rule_update", Boolean, nullable=False, default=False),
    Column("created_at", DateTime(timezone=True), nullable=False),
    Column("updated_at", DateTime(timezone=True), nullable=False),
    Index("ix_topical_correction_memory_type", "correction_type", "approved_for_rule_update"),
)

review_metadata.create_all(engine)
print("Human-in-the-Loop tables created/verified.")


Human-in-the-Loop tables created/verified.


## 5. Preflight counts


In [5]:
with engine.connect() as connection:
    counts = {
        "topics": connection.scalar(select(func.count()).select_from(topics)),
        "documents": connection.scalar(select(func.count()).select_from(documents)),
        "questions_all": connection.scalar(select(func.count()).select_from(questions)),
        "questions_scored": connection.scalar(select(func.count()).select_from(questions).where(questions.c.record_type == "scored_item")),
        "questions_context": connection.scalar(select(func.count()).select_from(questions).where(questions.c.record_type == "context")),
        "mark_scheme_entries": connection.scalar(select(func.count()).select_from(mark_scheme_entries)),
        "links": connection.scalar(select(func.count()).select_from(question_ms_links)),
        "open_issues": connection.scalar(select(func.count()).select_from(parsing_issues).where(parsing_issues.c.resolved.is_(False))),
        "retrieval_enabled": connection.scalar(select(func.count()).select_from(questions).where(questions.c.retrieval_enabled.is_(True))),
    }

preflight_df = pd.DataFrame([{"metric": k, "value": int(v or 0)} for k, v in counts.items()])
display(preflight_df)

if counts["questions_all"] == 0:
    raise RuntimeError("Notebook 02 data is missing.")


,metric,value
0,topics,35
1,documents,70
2,questions_all,942
3,questions_scored,823
4,questions_context,119
5,mark_scheme_entries,821
6,links,821
7,open_issues,20
8,retrieval_enabled,0


## 6. Load complete review snapshot


In [6]:
question_snapshot_query = (
    select(
        questions.c.id.label("question_id"),
        questions.c.question_uid,
        questions.c.pair_key,
        questions.c.topic_id,
        questions.c.question_document_id,
        questions.c.sequence_index,
        questions.c.question_number,
        questions.c.normalized_question_number,
        questions.c.occurrence_index,
        questions.c.main_question_number,
        questions.c.part_number,
        questions.c.record_type,
        questions.c.question_text,
        questions.c.context_text,
        questions.c.search_text,
        questions.c.raw_extracted_text,
        questions.c.marks,
        questions.c.page_start,
        questions.c.page_end,
        questions.c.has_visual,
        questions.c.visual_page_numbers,
        questions.c.has_code,
        questions.c.specification_scope,
        questions.c.is_legacy,
        questions.c.parse_warnings,
        questions.c.review_status.label("question_review_status"),
        questions.c.retrieval_enabled,
        questions.c.embedding_status,
        questions.c.is_active.label("question_is_active"),
        topics.c.pmt_topic_number.label("topic_number"),
        topics.c.pmt_topic_name.label("topic_name"),
        topics.c.pmt_subtopic_code.label("subtopic_code"),
        topics.c.pmt_subtopic_name.label("subtopic_name"),
        question_ms_links.c.id.label("link_id"),
        question_ms_links.c.match_method,
        question_ms_links.c.match_confidence,
        question_ms_links.c.marks_match,
        question_ms_links.c.validation_status.label("link_status"),
        question_ms_links.c.validation_warnings,
        mark_scheme_entries.c.id.label("ms_id"),
        mark_scheme_entries.c.maximum_marks,
        mark_scheme_entries.c.marking_guidance,
        mark_scheme_entries.c.review_status.label("ms_review_status"),
        mark_scheme_entries.c.parse_warnings.label("ms_parse_warnings"),
        mark_scheme_entries.c.is_legacy.label("ms_is_legacy"),
    )
    .select_from(
        questions
        .join(topics, questions.c.topic_id == topics.c.id)
        .outerjoin(question_ms_links, question_ms_links.c.question_id == questions.c.id)
        .outerjoin(mark_scheme_entries, mark_scheme_entries.c.id == question_ms_links.c.mark_scheme_entry_id)
    )
    .order_by(topics.c.pmt_topic_number, topics.c.pmt_subtopic_code, questions.c.sequence_index)
)

with engine.connect() as connection:
    snapshot_df = pd.read_sql(question_snapshot_query, connection)

print(f"Snapshot rows: {len(snapshot_df)}")
display(snapshot_df.head(10))


Snapshot rows: 942


,question_id,question_uid,pair_key,topic_id,question_document_id,sequence_index,question_number,normalized_question_number,occurrence_index,main_question_number,...,match_confidence,marks_match,link_status,validation_warnings,ms_id,maximum_marks,marking_guidance,ms_review_status,ms_parse_warnings,ms_is_legacy
0,d6b585dd-e3d3-497f-b9b2-75ea589fa120,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q1_1_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,1,01.1,1.1,1,1,...,1.0,True,auto_valid,[],4cdee56d-18af-4b74-bd23-ff87b1543afe,2.0,2 marks for AO1 (recall)\nA sequence/number/se...,auto_valid,"[alignment_method=exact_number_occurrence, ali...",False
1,d1a19dd3-1edb-46e8-93d4-026a3665c499,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q1_2_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,2,01.2,1.2,1,1,...,1.0,True,auto_valid,[],5a3c918f-2b4d-49f2-8803-03e7fe6833f2,3.0,3 marks for AO1 (recall)\nOne mark for each co...,auto_valid,"[alignment_method=exact_number_occurrence, ali...",False
2,0332d76f-c80e-4b93-95e2-7706bbd8dab6,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q2_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,3,02,2,1,2,...,1.0,True,auto_valid,[],398dc69e-0a71-4cac-ba85-53c029babc6e,7.0,7 marks for AO3 (program)\nIf CHAR_TO_CODE is ...,auto_valid,"[alignment_method=exact_number_occurrence, ali...",False
3,e7acf7bb-291e-4aa0-879a-6cda0b3162aa,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q3_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,4,03,3,1,3,...,1.0,True,auto_valid,[],5cfc2b96-dc38-4b51-a83e-d4bb48c3d617,8.0,8 marks for AO3 (program)\nDPT. For repeated e...,auto_valid,"[alignment_method=exact_number_occurrence, ali...",False
4,c58f37c6-12b6-4bd2-bb22-f3bb084bd8cb,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q4_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,5,04,4,1,4,...,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None
5,6ea35f5a-c317-46d2-b243-645ce9625f8c,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q4_1_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,6,04.1,4.1,1,4,...,1.0,True,auto_valid,[],fc1150cf-adb6-4b8f-a2fc-902450b151f9,3.0,3 marks for AO2 (apply)\n1 mark for C written ...,auto_valid,"[alignment_method=exact_number_occurrence, ali...",False
6,b03ac3af-d6bc-4805-9b3b-1e405867cd28,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q4_2_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,7,04.2,4.2,1,4,...,1.0,True,auto_valid,[],4f9613f1-2ba2-4e05-b1ad-98daa4febd0e,3.0,3 marks for AO2 (apply)\n1 mark for A written ...,auto_valid,"[alignment_method=exact_number_occurrence, ali...",False
7,6eab1253-bbd5-4382-b222-a156fd834bfe,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q4_3_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,8,04.3,4.3,1,4,...,1.0,True,auto_valid,[],b6320971-7c85-4244-a5e8-0f5d7da4f4a7,3.0,3 marks for AO2 (apply)\nIf any value is writt...,auto_valid,"[alignment_method=exact_number_occurrence, ali...",False
8,f17d176b-8946-4018-b692-c3cdc661b129,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q4_4_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,9,04.4,4.4,1,4,...,1.0,True,auto_valid,[],c9b4331c-a96c-462b-9af6-fc07a0051026,5.0,5 marks for AO3 (program)\nNote for mark C - D...,auto_valid,"[alignment_method=exact_number_occurrence, ali...",False
9,9db09e6b-17f8-458d-854c-1806b2267666,AQA_GCSE_CS_PMT_T1_1_1_PYTHON_QP_Q5_1_O01,AQA_GCSE_CS_PMT_T1_1_1_PYTHON,37945879-5e72-46a2-8529-13bc2a09ac1e,f63642ec-794e-465f-8623-5d6e9693be56,10,05.1,5.1,1,5,...,1.0,True,auto_valid,[],a781ee6a-734a-4eb1-82e3-4d7766d58d69,4.0,4 marks for AO2 (apply)\nMark A for totalSize ...,auto_valid,"[alignment_method=exact_number_

## 7. Automatic quality evaluation


In [7]:
def normalize_for_duplicate_check(value: str) -> str:
    value = (value or "").lower()
    value = re.sub(r"\s+", " ", value)
    value = re.sub(r"[^a-z0-9 ]+", "", value)
    return value.strip()

duplicate_counts = (
    snapshot_df.assign(
        duplicate_key=snapshot_df.apply(
            lambda row: normalize_for_duplicate_check(
                f"{row.get('context_text') or ''} {row.get('question_text') or ''}"
            ),
            axis=1,
        )
    )
    .groupby("duplicate_key", dropna=False)
    .size()
    .to_dict()
)

def check(code: str, passed: bool, severity: str, message: str, details: dict[str, Any] | None = None) -> dict[str, Any]:
    return {
        "check_code": code,
        "passed": bool(passed),
        "severity": severity,
        "message": message,
        "details": details or {},
    }

def evaluate_row(row: pd.Series) -> list[dict[str, Any]]:
    result: list[dict[str, Any]] = []

    question_number = str(row.get("question_number") or "").strip()
    question_text = str(row.get("question_text") or "").strip()
    context_text = str(row.get("context_text") or "").strip()
    raw_text = str(row.get("raw_extracted_text") or "").strip()
    is_scored = row.get("record_type") == "scored_item"

    result.append(check(
        "question_number_detected",
        bool(question_number and re.fullmatch(r"\d{1,2}(?:\.\d{1,2})?", question_number)),
        "critical",
        "Question number is present and valid.",
        {"value": question_number},
    ))

    marks_value = row.get("marks")
    marks_ok = (not is_scored) or (pd.notna(marks_value) and int(marks_value) > 0)
    result.append(check(
        "marks_detected",
        marks_ok,
        "critical",
        "A positive mark allocation exists for each scored item.",
        {"record_type": row.get("record_type"), "marks": make_json_safe(marks_value)},
    ))

    text_complete = len(question_text) >= MIN_QUESTION_TEXT_LENGTH and not question_text.endswith(("...", "…"))
    result.append(check(
        "question_text_complete",
        text_complete,
        "critical",
        "Question text appears complete.",
        {"length": len(question_text), "ending": question_text[-20:]},
    ))

    noise = [pattern for pattern in HEADER_FOOTER_PATTERNS if re.search(pattern, question_text, re.IGNORECASE)]
    result.append(check(
        "header_footer_removed",
        not noise,
        "warning",
        "No known header/footer noise remains.",
        {"matched_patterns": noise},
    ))

    link_found = (not is_scored) or pd.notna(row.get("link_id"))
    result.append(check(
        "qp_ms_link_found",
        link_found,
        "critical",
        "Scored question has a linked mark-scheme entry.",
        {"link_id": make_json_safe(row.get("link_id"))},
    ))

    duplicate_key = normalize_for_duplicate_check(f"{context_text} {question_text}")
    duplicate_total = int(duplicate_counts.get(duplicate_key, 0))
    result.append(check(
        "duplicate_question",
        not duplicate_key or duplicate_total <= 1,
        "warning",
        "Question text is not duplicated.",
        {"duplicate_count": duplicate_total},
    ))

    result.append(check(
        "suspiciously_short_question",
        not (is_scored and len(question_text) < SUSPICIOUS_SHORT_LENGTH),
        "warning",
        "Question text is not suspiciously short.",
        {"length": len(question_text)},
    ))

    has_code = bool(row.get("has_code"))
    code_ok = (not has_code) or len(raw_text) >= len(question_text) or bool(context_text)
    result.append(check(
        "code_context_retained",
        code_ok,
        "warning",
        "Code context appears retained.",
        {"has_code": has_code, "raw_length": len(raw_text), "context_length": len(context_text)},
    ))

    visual_pages = row.get("visual_page_numbers")
    if not isinstance(visual_pages, list):
        visual_pages = []
    has_visual = bool(row.get("has_visual"))
    visual_ok = (
        not has_visual
        or bool(visual_pages)
        or bool(re.search(r"\b(figure|diagram|table|graph|image)\b", f"{context_text} {question_text}", re.IGNORECASE))
    )
    result.append(check(
        "visual_context_retained",
        visual_ok,
        "warning",
        "Visual/table context is traceable.",
        {"has_visual": has_visual, "visual_pages": visual_pages},
    ))

    marks_consistent = (
        not is_scored
        or pd.isna(row.get("link_id"))
        or (
            pd.notna(row.get("marks"))
            and pd.notna(row.get("maximum_marks"))
            and int(row["marks"]) == int(row["maximum_marks"])
        )
    )
    result.append(check(
        "mark_allocation_consistent",
        marks_consistent,
        "critical",
        "QP and MS marks agree.",
        {"question_marks": make_json_safe(row.get("marks")), "ms_marks": make_json_safe(row.get("maximum_marks"))},
    ))

    exact_link = (
        not is_scored
        or pd.isna(row.get("link_id"))
        or row.get("match_method") == "exact_number_occurrence"
    )
    result.append(check(
        "exact_link_method",
        exact_link,
        "warning",
        "QP/MS link uses exact numbering.",
        {"match_method": make_json_safe(row.get("match_method")), "confidence": make_json_safe(row.get("match_confidence"))},
    ))

    result.append(check(
        "current_specification",
        not bool(row.get("is_legacy")),
        "warning",
        "Question is not explicitly legacy 8520.",
        {"specification_scope": make_json_safe(row.get("specification_scope")), "is_legacy": bool(row.get("is_legacy"))},
    ))

    guidance_ok = (
        not is_scored
        or pd.isna(row.get("link_id"))
        or len(str(row.get("marking_guidance") or "").strip()) >= 5
    )
    result.append(check(
        "marking_guidance_present",
        guidance_ok,
        "critical",
        "Linked mark-scheme guidance is present.",
        {"guidance_length": len(str(row.get("marking_guidance") or "").strip())},
    ))

    return result

check_rows: list[dict[str, Any]] = []
queue_rows: list[dict[str, Any]] = []

for _, row in snapshot_df.iterrows():
    row_checks = evaluate_row(row)
    failed = [item for item in row_checks if not item["passed"]]
    critical_count = sum(item["severity"] == "critical" for item in failed)
    warning_count = sum(item["severity"] == "warning" for item in failed)

    question_id = uuid.UUID(str(row["question_id"]))

    for item in row_checks:
        check_rows.append({
            "entity_type": "question",
            "entity_id": question_id,
            "pair_key": row["pair_key"],
            **item,
        })

    queue_rows.append({
        "queue_uid": f"question:{question_id}",
        "entity_type": "question",
        "entity_id": question_id,
        "pair_key": row["pair_key"],
        "priority": critical_count * 100 + warning_count * 10 + (25 if bool(row.get("is_legacy")) else 0),
        "status": "needs_review" if failed else "auto_approved",
        "reasons": [
            {
                "check_code": item["check_code"],
                "severity": item["severity"],
                "message": item["message"],
                "details": make_json_safe(item["details"]),
            }
            for item in failed
        ],
    })

with engine.connect() as connection:
    open_issue_rows = connection.execute(
        select(parsing_issues)
        .where(parsing_issues.c.resolved.is_(False))
        .order_by(parsing_issues.c.pair_key)
    ).mappings().all()

for issue in open_issue_rows:
    queue_rows.append({
        "queue_uid": f"issue:{issue['id']}",
        "entity_type": "issue",
        "entity_id": issue["id"],
        "pair_key": issue["pair_key"],
        "priority": {"critical": 150, "warning": 60, "info": 10}.get(issue["severity"], 10),
        "status": "needs_review",
        "reasons": [{
            "check_code": issue["issue_type"],
            "severity": issue["severity"],
            "message": issue["description"],
            "details": make_json_safe(issue["details"]),
        }],
    })

checks_df = pd.DataFrame(check_rows)
proposed_queue_df = pd.DataFrame(queue_rows)

print(f"Quality checks: {len(checks_df)}")
print(f"Queue items:    {len(proposed_queue_df)}")
display(proposed_queue_df.sort_values(["status", "priority"], ascending=[False, False]).head(30))


Quality checks: 12246
Queue items:    962


,queue_uid,entity_type,entity_id,pair_key,priority,status,reasons
944,issue:c20ada2a-1d3a-46af-804c-242d1ccb2bbc,issue,c20ada2a-1d3a-46af-804c-242d1ccb2bbc,AQA_GCSE_CS_PMT_T1_1_3_PYTHON,150,needs_review,[{'check_code': 'questions_without_ms_candidat...
945,issue:adce6744-ae85-40a9-801c-9fb18e75d2be,issue,adce6744-ae85-40a9-801c-9fb18e75d2be,AQA_GCSE_CS_PMT_T1_1_3_PYTHON,150,needs_review,"[{'check_code': 'unlinked_questions', 'severit..."
946,issue:65056c24-dd20-48fd-bb2d-70c33cb4281c,issue,65056c24-dd20-48fd-bb2d-70c33cb4281c,AQA_GCSE_CS_PMT_T1_1_4_PYTHON,150,needs_review,"[{'check_code': 'linked_marks_mismatch', 'seve..."
948,issue:2c458cf9-2b5e-4500-b8ce-48dd728fbe7a,issue,2c458cf9-2b5e-4500-b8ce-48dd728fbe7a,AQA_GCSE_CS_PMT_T2_2_02_PYTHON,150,needs_review,"[{'check_code': 'linked_marks_mismatch', 'seve..."
955,issue:89331ad5-fe66-44ec-b761-7ffd9b27496b,issue,89331ad5-fe66-44ec-b761-7ffd9b27496b,AQA_GCSE_CS_PMT_T3_3_8_THEORY,150,needs_review,"[{'check_code': 'linked_marks_mismatch', 'seve..."
956,issue:f4b53f75-9e03-4878-b246-30c1a810d1d1,issue,f4b53f75-9e03-4878-b246-30c1a810d1d1,AQA_GCSE_CS_PMT_T5_5_THEORY,150,needs_review,[{'check_code': 'questions_without_ms_candidat...
958,issue:ce06ba23-e9d9-4b93-aa4a-3cbcb34d4894,issue,ce06ba23-e9d9-4b93-aa4a-3cbcb34d4894,AQA_GCSE_CS_PMT_T5_5_THEORY,150,needs_review,"[{'check_code': 'unlinked_questions', 'severit..."
85,question:7dab3619-89bb-4a73-8eb9-704fd98d4561,question,7dab3619-89bb-4a73-8eb9-704fd98d4561,AQA_GCSE_CS_PMT_T1_1_4_PYTHON,110,needs_review,"[{'check_code': 'duplicate_question', 'severit..."
139,question:5cfe6914-db99-47fc-93c7-41832cabe7e3,question,5cfe6914-db99-47fc-93c7-41832cabe7e3,AQA_GCSE_CS_PMT_T2_2_02_PYTHON,110,needs_review,"[{'check_code': 'duplicate_question', 'severit..."
877,question:ebecd8c4-3a91-4558-b0a8-f912a8bddd05,question,ebecd8c4-3a91-4558-b0a8-f912a8bddd05,AQA_GCSE_CS_PMT_T5_5_THEORY,110,needs_review,"[{'check_code': 'qp_ms_link_found', 'severity'..."


## 8. Save the quality run and persistent review queue


In [8]:
def upsert_queue_item(session: Session, run_id: uuid.UUID, item: dict[str, Any]) -> None:
    now = utc_now()
    existing = session.execute(
        select(review_queue).where(review_queue.c.queue_uid == item["queue_uid"])
    ).mappings().first()

    protected = {"approved", "corrected", "rejected", "legacy_disabled"}
    next_status = (
        existing["status"]
        if existing and existing["status"] in protected and not RESET_EXISTING_DECISIONS
        else item["status"]
    )

    statement = (
        pg_insert(review_queue)
        .values(
            id=uuid.uuid4(),
            queue_uid=item["queue_uid"],
            quality_run_id=run_id,
            entity_type=item["entity_type"],
            entity_id=item["entity_id"],
            pair_key=item["pair_key"],
            priority=int(item["priority"]),
            status=next_status,
            reasons=make_json_safe(item["reasons"]),
            created_at=now,
            updated_at=now,
        )
        .on_conflict_do_update(
            index_elements=[review_queue.c.queue_uid],
            set_={
                "quality_run_id": run_id,
                "pair_key": item["pair_key"],
                "priority": int(item["priority"]),
                "status": next_status,
                "reasons": make_json_safe(item["reasons"]),
                "updated_at": now,
            },
        )
    )
    session.execute(statement)

quality_run_id: uuid.UUID | None = None

if SAVE_EVALUATION_TO_DB:
    quality_run_id = uuid.uuid4()
    now = utc_now()

    try:
        with Session(engine) as session:
            session.execute(insert(quality_runs).values(
                id=quality_run_id,
                evaluator_version=EVALUATOR_VERSION,
                status="running",
                started_at=now,
                counts={},
            ))

            for item in check_rows:
                session.execute(insert(quality_checks).values(
                    id=uuid.uuid4(),
                    run_id=quality_run_id,
                    entity_type=item["entity_type"],
                    entity_id=item["entity_id"],
                    pair_key=item["pair_key"],
                    check_code=item["check_code"],
                    passed=bool(item["passed"]),
                    severity=item["severity"],
                    message=item["message"],
                    details=make_json_safe(item["details"]),
                    created_at=utc_now(),
                ))

            for item in queue_rows:
                upsert_queue_item(session, quality_run_id, item)

            counts_payload = {
                "question_checks": len(check_rows),
                "queue_items": len(queue_rows),
                "auto_approved": sum(item["status"] == "auto_approved" for item in queue_rows),
                "needs_review": sum(item["status"] == "needs_review" for item in queue_rows),
                "open_parsing_issues": len(open_issue_rows),
            }

            session.execute(
                update(quality_runs)
                .where(quality_runs.c.id == quality_run_id)
                .values(status="completed", completed_at=utc_now(), counts=counts_payload)
            )
            session.commit()

        print(f"Quality run saved: {quality_run_id}")

    except Exception as error:
        with Session(engine) as session:
            session.execute(
                update(quality_runs)
                .where(quality_runs.c.id == quality_run_id)
                .values(status="failed", completed_at=utc_now(), error_message=str(error))
            )
            session.commit()
        raise
else:
    print("Evaluation not saved.")


DataError: (psycopg.errors.InvalidTextRepresentation) invalid input syntax for type json
DETAIL:  Token "NaN" is invalid.
CONTEXT:  JSON data, line 1: {"record_type": "context", "marks": NaN...
unnamed portal parameter $10 = '...'
[SQL: INSERT INTO assessment_topical_quality_checks (id, run_id, entity_type, entity_id, pair_key, check_code, passed, severity, message, details, created_at) VALUES (%(id)s::UUID, %(run_id)s::UUID, %(entity_type)s::VARCHAR, %(entity_id)s::UUID, %(pair_key)s::VARCHAR, %(check_code)s::VARCHAR, %(passed)s, %(severity)s::VARCHAR, %(message)s::VARCHAR, %(details)s::JSONB, %(created_at)s::TIMESTAMP WITH TIME ZONE)]
[parameters: {'id': UUID('6a9e58ed-cfb0-4bf7-bea2-b357b8ba1348'), 'run_id': UUID('0bed3d63-81b9-42ab-9253-d2783998ef57'), 'entity_type': 'question', 'entity_id': UUID('c58f37c6-12b6-4bd2-bb22-f3bb084bd8cb'), 'pair_key': 'AQA_GCSE_CS_PMT_T1_1_1_PYTHON', 'check_code': 'marks_detected', 'passed': True, 'severity': 'critical', 'message': 'A positive mark al

## 9. Queue inspection helpers


In [ ]:
def load_review_queue(statuses: list[str] | None = None) -> pd.DataFrame:
    query = select(review_queue)
    if statuses:
        query = query.where(review_queue.c.status.in_(statuses))
    query = query.order_by(review_queue.c.priority.desc(), review_queue.c.pair_key)
    with engine.connect() as connection:
        return pd.read_sql(query, connection)

def get_review_bundle(queue_id: str | uuid.UUID) -> dict[str, Any]:
    queue_uuid = uuid.UUID(str(queue_id))
    with engine.connect() as connection:
        queue_item = connection.execute(
            select(review_queue).where(review_queue.c.id == queue_uuid)
        ).mappings().first()
        if queue_item is None:
            raise ValueError(f"Queue item not found: {queue_uuid}")

        bundle: dict[str, Any] = {"queue": dict(queue_item)}

        if queue_item["entity_type"] == "question":
            row = connection.execute(
                question_snapshot_query.where(questions.c.id == queue_item["entity_id"])
            ).mappings().first()
            bundle["question"] = dict(row) if row else None
        else:
            row = connection.execute(
                select(parsing_issues).where(parsing_issues.c.id == queue_item["entity_id"])
            ).mappings().first()
            bundle["issue"] = dict(row) if row else None

        return make_json_safe(bundle)

def display_review_bundle(queue_id: str | uuid.UUID) -> None:
    bundle = get_review_bundle(queue_id)
    item = bundle["queue"]

    display(HTML(
        "<h3>Review Item</h3>"
        f"<b>Type:</b> {item['entity_type']}<br>"
        f"<b>Pair:</b> {item['pair_key']}<br>"
        f"<b>Priority:</b> {item['priority']}<br>"
        f"<b>Status:</b> {item['status']}<br>"
    ))
    display(pd.DataFrame(item["reasons"]))

    if item["entity_type"] == "question":
        q = bundle["question"]
        display(HTML(
            f"<h4>{q['topic_name']} - {q['subtopic_name']}</h4>"
            f"<b>Question:</b> {q['question_number']}<br>"
            f"<b>Marks:</b> {q['marks']}<br>"
            f"<b>MS marks:</b> {q['maximum_marks']}<br>"
            f"<b>Match method:</b> {q['match_method']}<br>"
            f"<b>Confidence:</b> {q['match_confidence']}<br>"
            f"<b>Legacy:</b> {q['is_legacy']}<br><br>"
            "<b>Question text</b>"
            f"<pre style='white-space:pre-wrap'>{q['question_text']}</pre>"
            "<b>Context</b>"
            f"<pre style='white-space:pre-wrap'>{q['context_text']}</pre>"
            "<b>Marking guidance</b>"
            f"<pre style='white-space:pre-wrap'>{q['marking_guidance']}</pre>"
        ))
    else:
        display(pd.DataFrame([bundle["issue"]]).T)

queue_df = load_review_queue()
display(
    queue_df.groupby(["entity_type", "status"], dropna=False)
    .size().reset_index(name="total")
)
display(load_review_queue(["needs_review"]).head(50))


## 10. Audit, correction memory, and locked-row helpers


In [ ]:
def calculate_content_hash(payload: dict[str, Any]) -> str:
    encoded = json.dumps(make_json_safe(payload), ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

def get_queue_item_for_update(session: Session, queue_id: str | uuid.UUID) -> dict[str, Any]:
    queue_uuid = uuid.UUID(str(queue_id))
    item = session.execute(
        select(review_queue).where(review_queue.c.id == queue_uuid).with_for_update()
    ).mappings().first()
    if item is None:
        raise ValueError(f"Queue item not found: {queue_uuid}")
    return dict(item)

def add_review(
    session: Session,
    queue_item: dict[str, Any],
    action: str,
    reviewer: str,
    note: str,
    old_values: dict[str, Any],
    new_values: dict[str, Any],
) -> None:
    session.execute(insert(human_reviews).values(
        id=uuid.uuid4(),
        queue_id=queue_item["id"],
        entity_type=queue_item["entity_type"],
        entity_id=queue_item["entity_id"],
        action=action,
        reviewer=reviewer,
        note=note,
        old_values=make_json_safe(old_values),
        new_values=make_json_safe(new_values),
        created_at=utc_now(),
    ))

def add_memory(
    session: Session,
    correction_type: str,
    pair_key: str,
    entity_id: uuid.UUID,
    original_value: dict[str, Any],
    corrected_value: dict[str, Any],
    reviewer: str,
) -> None:
    memory_key = calculate_content_hash({
        "correction_type": correction_type,
        "pair_key": pair_key,
        "original": original_value,
        "corrected": corrected_value,
    })
    now = utc_now()
    session.execute(
        pg_insert(correction_memory)
        .values(
            id=uuid.uuid4(),
            memory_key=memory_key,
            correction_type=correction_type,
            pair_key=pair_key,
            source_entity_id=entity_id,
            original_value=make_json_safe(original_value),
            corrected_value=make_json_safe(corrected_value),
            reviewer=reviewer,
            confirmations=1,
            approved_for_rule_update=False,
            created_at=now,
            updated_at=now,
        )
        .on_conflict_do_update(
            index_elements=[correction_memory.c.memory_key],
            set_={
                "confirmations": correction_memory.c.confirmations + 1,
                "reviewer": reviewer,
                "updated_at": now,
            },
        )
    )


## 11. Approve, reject, and disable legacy items


In [ ]:
def approve_queue_item(
    queue_id: str | uuid.UUID,
    reviewer: str = DEFAULT_REVIEWER,
    note: str = "",
    action: str = "approve",
) -> None:
    with Session(engine) as session:
        item = get_queue_item_for_update(session, queue_id)

        if item["entity_type"] == "issue":
            issue = session.execute(
                select(parsing_issues)
                .where(parsing_issues.c.id == item["entity_id"])
                .with_for_update()
            ).mappings().first()
            if issue is None:
                raise ValueError("Parsing issue not found.")

            updates = {
                "resolved": True,
                "resolution_note": note,
                "resolved_by": reviewer,
                "resolved_at": utc_now(),
            }
            session.execute(update(parsing_issues).where(parsing_issues.c.id == issue["id"]).values(**updates))
            add_review(session, item, "resolve_issue", reviewer, note, {"issue": dict(issue)}, {"issue": updates})

        elif item["entity_type"] == "question":
            q = session.execute(
                select(questions)
                .where(questions.c.id == item["entity_id"])
                .with_for_update()
            ).mappings().first()
            if q is None:
                raise ValueError("Question not found.")
            if bool(q["is_legacy"]):
                raise ValueError("Legacy item cannot be approved. Use legacy_disable_queue_item().")

            link = session.execute(
                select(question_ms_links)
                .where(question_ms_links.c.question_id == q["id"])
                .with_for_update()
            ).mappings().first()

            if q["record_type"] == "scored_item" and link is None:
                raise ValueError("Scored question has no MS link.")

            q_updates = {
                "review_status": "human_approved",
                "retrieval_enabled": q["record_type"] == "scored_item",
                "embedding_status": "ready_for_indexing" if q["record_type"] == "scored_item" else "context_only",
                "is_active": True,
                "updated_at": utc_now(),
            }
            session.execute(update(questions).where(questions.c.id == q["id"]).values(**q_updates))

            if link:
                session.execute(
                    update(question_ms_links)
                    .where(question_ms_links.c.id == link["id"])
                    .values(validation_status="human_approved")
                )
                session.execute(
                    update(mark_scheme_entries)
                    .where(mark_scheme_entries.c.id == link["mark_scheme_entry_id"])
                    .values(review_status="human_approved", updated_at=utc_now())
                )

            add_review(session, item, action, reviewer, note, {"question": dict(q)}, {"question": q_updates})

        else:
            raise ValueError(f"Unsupported entity type: {item['entity_type']}")

        session.execute(
            update(review_queue)
            .where(review_queue.c.id == item["id"])
            .values(status="approved", assigned_to=reviewer, reviewer_note=note, updated_at=utc_now())
        )
        session.commit()

    print(f"Approved/resolved: {queue_id}")

def reject_queue_item(
    queue_id: str | uuid.UUID,
    reviewer: str = DEFAULT_REVIEWER,
    note: str = "",
    legacy_disable: bool = False,
) -> None:
    if not note.strip():
        raise ValueError("A rejection/disable note is required.")

    with Session(engine) as session:
        item = get_queue_item_for_update(session, queue_id)
        if item["entity_type"] != "question":
            raise ValueError("Reject/disable supports question items only.")

        q = session.execute(
            select(questions)
            .where(questions.c.id == item["entity_id"])
            .with_for_update()
        ).mappings().first()
        if q is None:
            raise ValueError("Question not found.")

        q_updates = {
            "review_status": "rejected",
            "retrieval_enabled": False,
            "embedding_status": "legacy_disabled" if legacy_disable else "rejected",
            "is_active": False,
            "updated_at": utc_now(),
        }
        session.execute(update(questions).where(questions.c.id == q["id"]).values(**q_updates))

        link = session.execute(
            select(question_ms_links)
            .where(question_ms_links.c.question_id == q["id"])
            .with_for_update()
        ).mappings().first()

        if link:
            session.execute(
                update(question_ms_links)
                .where(question_ms_links.c.id == link["id"])
                .values(validation_status="rejected")
            )
            session.execute(
                update(mark_scheme_entries)
                .where(mark_scheme_entries.c.id == link["mark_scheme_entry_id"])
                .values(review_status="rejected", is_active=False, updated_at=utc_now())
            )

        status = "legacy_disabled" if legacy_disable else "rejected"
        action = "legacy_disable" if legacy_disable else "reject"

        session.execute(
            update(review_queue)
            .where(review_queue.c.id == item["id"])
            .values(status=status, assigned_to=reviewer, reviewer_note=note, updated_at=utc_now())
        )
        add_review(session, item, action, reviewer, note, {"question": dict(q)}, {"question": q_updates})
        session.commit()

    print(f"{status}: {queue_id}")

def legacy_disable_queue_item(
    queue_id: str | uuid.UUID,
    reviewer: str = DEFAULT_REVIEWER,
    note: str = "Legacy 8520 item disabled from retrieval.",
) -> None:
    reject_queue_item(queue_id, reviewer=reviewer, note=note, legacy_disable=True)


## 12. Correct a question and/or linked mark scheme


In [ ]:
ALLOWED_QUESTION_CORRECTIONS = {"question_text", "context_text", "marks", "has_code", "has_visual"}
ALLOWED_MS_CORRECTIONS = {"maximum_marks", "marking_guidance"}

def correct_question_and_ms(
    queue_id: str | uuid.UUID,
    question_corrections: dict[str, Any] | None = None,
    ms_corrections: dict[str, Any] | None = None,
    reviewer: str = DEFAULT_REVIEWER,
    note: str = "",
) -> None:
    question_corrections = question_corrections or {}
    ms_corrections = ms_corrections or {}

    if not note.strip():
        raise ValueError("Correction note is required.")
    if not question_corrections and not ms_corrections:
        raise ValueError("Provide at least one correction.")
    if set(question_corrections) - ALLOWED_QUESTION_CORRECTIONS:
        raise ValueError(f"Unsupported question fields: {sorted(set(question_corrections)-ALLOWED_QUESTION_CORRECTIONS)}")
    if set(ms_corrections) - ALLOWED_MS_CORRECTIONS:
        raise ValueError(f"Unsupported MS fields: {sorted(set(ms_corrections)-ALLOWED_MS_CORRECTIONS)}")

    with Session(engine) as session:
        item = get_queue_item_for_update(session, queue_id)
        if item["entity_type"] != "question":
            raise ValueError("Correction supports question items only.")

        q = session.execute(
            select(questions).where(questions.c.id == item["entity_id"]).with_for_update()
        ).mappings().first()
        if q is None:
            raise ValueError("Question not found.")
        if bool(q["is_legacy"]):
            raise ValueError("Legacy item should be disabled rather than promoted.")

        link = session.execute(
            select(question_ms_links)
            .where(question_ms_links.c.question_id == q["id"])
            .with_for_update()
        ).mappings().first()

        ms = None
        if link:
            ms = session.execute(
                select(mark_scheme_entries)
                .where(mark_scheme_entries.c.id == link["mark_scheme_entry_id"])
                .with_for_update()
            ).mappings().first()

        if ms_corrections and ms is None:
            raise ValueError("MS correction requested but no linked MS entry exists.")

        old_values = {"question": dict(q), "mark_scheme": dict(ms) if ms else None}

        q_updates = dict(question_corrections)
        new_question_text = str(q_updates.get("question_text", q["question_text"]))
        new_context_text = str(q_updates.get("context_text", q["context_text"]))
        new_question_marks = int(q_updates.get("marks", q["marks"])) if q["marks"] is not None else q_updates.get("marks")

        topic = session.execute(select(topics).where(topics.c.id == q["topic_id"])).mappings().first()

        q_updates.update({
            "search_text": (
                f"Topic: {topic['pmt_topic_name']}\n"
                f"Subtopic: {topic['pmt_subtopic_name']}\n\n"
                f"{new_context_text}\n\n{new_question_text}"
            ).strip(),
            "review_status": "human_corrected",
            "retrieval_enabled": q["record_type"] == "scored_item",
            "embedding_status": "ready_for_indexing" if q["record_type"] == "scored_item" else "context_only",
            "is_active": True,
            "content_hash": calculate_content_hash({
                "question_text": new_question_text,
                "context_text": new_context_text,
                "marks": new_question_marks,
            }),
            "updated_at": utc_now(),
        })

        session.execute(update(questions).where(questions.c.id == q["id"]).values(**q_updates))

        ms_updates: dict[str, Any] = {}
        if ms:
            ms_updates = dict(ms_corrections)
            new_ms_marks = int(ms_updates.get("maximum_marks", ms["maximum_marks"]))
            new_guidance = str(ms_updates.get("marking_guidance", ms["marking_guidance"]))

            if new_question_marks is not None and new_question_marks != new_ms_marks:
                raise ValueError("Corrected QP and MS marks still do not match.")

            ms_updates.update({
                "review_status": "human_corrected",
                "is_active": True,
                "content_hash": calculate_content_hash({
                    "marking_guidance": new_guidance,
                    "maximum_marks": new_ms_marks,
                }),
                "updated_at": utc_now(),
            })

            session.execute(
                update(mark_scheme_entries)
                .where(mark_scheme_entries.c.id == ms["id"])
                .values(**ms_updates)
            )
            session.execute(
                update(question_ms_links)
                .where(question_ms_links.c.id == link["id"])
                .values(
                    marks_match=True,
                    validation_status="human_approved",
                    validation_warnings=[],
                )
            )

        session.execute(
            update(review_queue)
            .where(review_queue.c.id == item["id"])
            .values(status="corrected", assigned_to=reviewer, reviewer_note=note, updated_at=utc_now())
        )

        new_values = {"question": q_updates, "mark_scheme": ms_updates}
        add_review(session, item, "correct", reviewer, note, old_values, new_values)
        add_memory(session, "question_or_ms_correction", item["pair_key"], q["id"], old_values, new_values, reviewer)
        session.commit()

    print(f"Corrected and promoted: {queue_id}")


## 13. Create a missing MS entry and link it manually


In [ ]:
def create_manual_ms_entry_and_link(
    queue_id: str | uuid.UUID,
    marking_guidance: str,
    maximum_marks: int,
    reviewer: str = DEFAULT_REVIEWER,
    note: str = "",
) -> None:
    if not marking_guidance.strip():
        raise ValueError("Marking guidance is required.")
    if maximum_marks <= 0:
        raise ValueError("maximum_marks must be positive.")
    if not note.strip():
        raise ValueError("Manual-link note is required.")

    with Session(engine) as session:
        item = get_queue_item_for_update(session, queue_id)
        if item["entity_type"] != "question":
            raise ValueError("Manual link supports question items only.")

        q = session.execute(
            select(questions).where(questions.c.id == item["entity_id"]).with_for_update()
        ).mappings().first()
        if q is None:
            raise ValueError("Question not found.")

        existing_link = session.execute(
            select(question_ms_links).where(question_ms_links.c.question_id == q["id"])
        ).mappings().first()
        if existing_link:
            raise ValueError("Question already has an MS link.")

        if int(q["marks"]) != int(maximum_marks):
            raise ValueError("Manual MS marks do not match question marks.")

        ms_document = session.execute(
            select(documents).where(
                documents.c.pair_key == q["pair_key"],
                documents.c.document_type == "mark_scheme",
            )
        ).mappings().first()
        if ms_document is None:
            raise ValueError("Matching MS document not found.")

        next_sequence = session.scalar(
            select(func.coalesce(func.max(mark_scheme_entries.c.sequence_index), 0) + 1)
            .where(mark_scheme_entries.c.mark_scheme_document_id == ms_document["id"])
        ) or 1

        ms_id = uuid.uuid4()
        link_id = uuid.uuid4()
        manual_uid = f"{q['pair_key']}_MANUAL_MS_{q['normalized_question_number']}_{ms_id.hex[:8]}"

        session.execute(insert(mark_scheme_entries).values(
            id=ms_id,
            mark_scheme_uid=manual_uid,
            pair_key=q["pair_key"],
            topic_id=q["topic_id"],
            mark_scheme_document_id=ms_document["id"],
            sequence_index=int(next_sequence),
            question_number=q["question_number"],
            normalized_question_number=q["normalized_question_number"],
            occurrence_index=q["occurrence_index"],
            main_question_number=q["main_question_number"],
            part_number=q["part_number"],
            maximum_marks=int(maximum_marks),
            marking_guidance=marking_guidance.strip(),
            marking_points=[line.strip() for line in marking_guidance.splitlines() if line.strip()],
            acceptable_answers=[],
            rejected_answers=[],
            additional_guidance=[],
            assessment_objectives=[],
            page_start=q["page_start"],
            page_end=q["page_end"],
            raw_extracted_text=marking_guidance.strip(),
            specification_scope=q["specification_scope"],
            is_legacy=bool(q["is_legacy"]),
            parse_warnings=["human_created_mark_scheme_entry"],
            review_status="human_corrected",
            content_hash=calculate_content_hash({"guidance": marking_guidance, "marks": maximum_marks}),
            parse_version="human-review-v1",
            is_active=True,
            created_at=utc_now(),
            updated_at=utc_now(),
        ))

        session.execute(insert(question_ms_links).values(
            id=link_id,
            pair_key=q["pair_key"],
            question_id=q["id"],
            mark_scheme_entry_id=ms_id,
            match_method="human_manual_link",
            match_confidence=1.0,
            marks_match=True,
            validation_status="human_approved",
            validation_warnings=[],
            created_at=utc_now(),
        ))

        q_updates = {
            "review_status": "human_corrected",
            "retrieval_enabled": True,
            "embedding_status": "ready_for_indexing",
            "is_active": True,
            "updated_at": utc_now(),
        }
        session.execute(update(questions).where(questions.c.id == q["id"]).values(**q_updates))
        session.execute(
            update(review_queue)
            .where(review_queue.c.id == item["id"])
            .values(status="corrected", assigned_to=reviewer, reviewer_note=note, updated_at=utc_now())
        )

        add_review(
            session, item, "manual_link", reviewer, note,
            {"question": dict(q), "link": None},
            {"question": q_updates, "manual_ms_id": ms_id, "link_id": link_id},
        )
        add_memory(
            session, "manual_ms_creation_and_link", item["pair_key"], q["id"],
            {"link": None},
            {"maximum_marks": maximum_marks, "marking_guidance": marking_guidance},
            reviewer,
        )
        session.commit()

    print(f"Manual MS entry created and linked: {queue_id}")


## 14. Guarded bulk promotion of automatic passes


In [ ]:
CONFIRM_BULK_AUTO_APPROVAL = False

def bulk_promote_auto_approved(
    reviewer: str = DEFAULT_REVIEWER,
    note: str = "Bulk-approved after automatic checks.",
) -> dict[str, int]:
    if not CONFIRM_BULK_AUTO_APPROVAL:
        raise RuntimeError(
            "Inspect auto-approved records, then set CONFIRM_BULK_AUTO_APPROVAL=True."
        )

    result = {"scored_promoted": 0, "context_approved": 0, "legacy_skipped": 0, "failed": 0}
    auto_df = load_review_queue(["auto_approved"])

    for _, item in auto_df.iterrows():
        if item["entity_type"] != "question":
            continue
        try:
            bundle = get_review_bundle(item["id"])
            q = bundle["question"]

            if q["is_legacy"]:
                result["legacy_skipped"] += 1
                continue

            approve_queue_item(
                item["id"],
                reviewer=reviewer,
                note=note,
                action="bulk_approve",
            )

            if q["record_type"] == "scored_item":
                result["scored_promoted"] += 1
            else:
                result["context_approved"] += 1

        except Exception as error:
            result["failed"] += 1
            print(f"Failed {item['id']}: {error}")

    print(result)
    return result

print("Bulk promotion locked.")


## 15. Interactive review widget


In [ ]:
def build_review_widget(statuses: list[str] | None = None):
    statuses = statuses or ["needs_review"]
    current_df = load_review_queue(statuses)

    if current_df.empty:
        display(HTML("<b>No queue items for these statuses.</b>"))
        return None

    selector = widgets.Dropdown(
        options=[
            (
                f"{row.entity_type} | {row.pair_key} | P{row.priority} | {str(row.id)[:8]}",
                str(row.id),
            )
            for row in current_df.itertuples()
        ],
        description="Review:",
        layout=widgets.Layout(width="95%"),
    )

    reviewer_input = widgets.Text(
        value=DEFAULT_REVIEWER,
        description="Reviewer:",
        layout=widgets.Layout(width="95%"),
    )

    note_input = widgets.Textarea(
        value="",
        description="Note:",
        layout=widgets.Layout(width="95%", height="90px"),
    )

    approve_button = widgets.Button(description="Approve", button_style="success")
    reject_button = widgets.Button(description="Reject", button_style="danger")
    legacy_button = widgets.Button(description="Disable Legacy", button_style="warning")
    refresh_button = widgets.Button(description="Refresh")
    output = widgets.Output()

    def render(*_):
        with output:
            clear_output(wait=True)
            display_review_bundle(selector.value)

    def act(kind: str):
        with output:
            clear_output(wait=True)
            try:
                reviewer = reviewer_input.value.strip() or DEFAULT_REVIEWER
                note = note_input.value.strip()

                if kind == "approve":
                    approve_queue_item(selector.value, reviewer=reviewer, note=note)
                elif kind == "reject":
                    reject_queue_item(selector.value, reviewer=reviewer, note=note)
                else:
                    legacy_disable_queue_item(
                        selector.value,
                        reviewer=reviewer,
                        note=note or "Legacy 8520 item disabled from retrieval.",
                    )

                print("Action saved. Rerun the widget cell to refresh the list.")
            except Exception as error:
                print(f"{type(error).__name__}: {error}")

    selector.observe(render, names="value")
    approve_button.on_click(lambda _: act("approve"))
    reject_button.on_click(lambda _: act("reject"))
    legacy_button.on_click(lambda _: act("legacy"))
    refresh_button.on_click(lambda _: render())

    ui = widgets.VBox([
        selector,
        reviewer_input,
        note_input,
        widgets.HBox([approve_button, reject_button, legacy_button, refresh_button]),
        output,
    ])

    display(ui)
    render()
    return ui

review_widget = build_review_widget(["needs_review"])


## 16. Typical correction commands

```python
# Marks mismatch
correct_question_and_ms(
    "QUEUE-UUID",
    question_corrections={"marks": 4},
    ms_corrections={"maximum_marks": 4},
    reviewer="Rida",
    note="Verified against original QP/MS pages.",
)

# Incomplete text
correct_question_and_ms(
    "QUEUE-UUID",
    question_corrections={"question_text": "Complete corrected text..."},
    reviewer="Rida",
    note="Restored missing continuation text.",
)

# True missing MS entry
create_manual_ms_entry_and_link(
    "QUEUE-UUID",
    marking_guidance="One mark for...",
    maximum_marks=2,
    reviewer="Rida",
    note="Verified manually from source MS.",
)

# False-positive question
reject_queue_item(
    "QUEUE-UUID",
    reviewer="Rida",
    note="Numeric table row incorrectly parsed as a question.",
)
```


## 17. Post-review report and exports


In [ ]:
def post_review_report():
    with engine.connect() as connection:
        question_status_df = pd.read_sql(
            select(
                questions.c.review_status,
                questions.c.retrieval_enabled,
                questions.c.embedding_status,
                func.count().label("total"),
            )
            .group_by(
                questions.c.review_status,
                questions.c.retrieval_enabled,
                questions.c.embedding_status,
            )
            .order_by(questions.c.review_status),
            connection,
        )

        queue_status_df = pd.read_sql(
            select(
                review_queue.c.entity_type,
                review_queue.c.status,
                func.count().label("total"),
            )
            .group_by(review_queue.c.entity_type, review_queue.c.status)
            .order_by(review_queue.c.entity_type, review_queue.c.status),
            connection,
        )

        issue_status_df = pd.read_sql(
            select(
                parsing_issues.c.severity,
                parsing_issues.c.resolved,
                func.count().label("total"),
            )
            .group_by(parsing_issues.c.severity, parsing_issues.c.resolved)
            .order_by(parsing_issues.c.severity, parsing_issues.c.resolved),
            connection,
        )

    return question_status_df, queue_status_df, issue_status_df

question_status_df, queue_status_df, issue_status_df = post_review_report()
display(question_status_df)
display(queue_status_df)
display(issue_status_df)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

with engine.connect() as connection:
    queue_export_df = pd.read_sql(select(review_queue).order_by(review_queue.c.status, review_queue.c.priority.desc()), connection)
    audit_export_df = pd.read_sql(select(human_reviews).order_by(human_reviews.c.created_at), connection)
    memory_export_df = pd.read_sql(select(correction_memory).order_by(correction_memory.c.updated_at.desc()), connection)

queue_path = OUTPUT_DIR / f"agent2_human_review_queue_{timestamp}.csv"
audit_path = OUTPUT_DIR / f"agent2_human_review_audit_{timestamp}.csv"
memory_path = OUTPUT_DIR / f"agent2_correction_memory_{timestamp}.csv"

queue_export_df.to_csv(queue_path, index=False)
audit_export_df.to_csv(audit_path, index=False)
memory_export_df.to_csv(memory_path, index=False)

print(queue_path)
print(audit_path)
print(memory_path)


# Notebook 03 completion criteria

```text
automatic quality run saved
all critical needs_review records handled
2 unlinked questions corrected or rejected
3 marks mismatches corrected
2 non-exact alignments approved/corrected
legacy 8520 questions disabled
critical parsing issues resolved
approved scored questions:
    retrieval_enabled = True
    embedding_status = ready_for_indexing
rejected questions:
    retrieval_enabled = False
correction memory populated
```

## Next notebook

```text
04_question_embeddings_and_qdrant_indexing.ipynb
```

Notebook 04 should index only:

```sql
WHERE record_type = 'scored_item'
  AND review_status IN ('human_approved', 'human_corrected')
  AND retrieval_enabled = TRUE
  AND is_active = TRUE
  AND is_legacy = FALSE
  AND embedding_status = 'ready_for_indexing'
```
